In [1]:
import asyncio
import os
from collections.abc import Sequence
from typing import Any

from dotenv import load_dotenv
from agent_framework import AgentSession, HistoryProvider, Message
from agent_framework.openai import OpenAIChatClient
from azure.identity.aio import AzureCliCredential

In [2]:
load_dotenv(override=True)

azure_endpoint = os.getenv("AZURE_OPENAI_ENDPOINT")
model = os.getenv("AZURE_OPENAI_RESPONSES_DEPLOYMENT_NAME")

print("Azure OpenAI Endpoint: ", azure_endpoint)
print("Model: ", model)

Azure OpenAI Endpoint:  https://ramkumar-foundry-v19.services.ai.azure.com
Model:  gpt-4o


In [3]:
class CustomHistoryProvider(HistoryProvider):
    def __init__(self) -> None:
        super().__init__("custom-history")
        self._storage: dict[str, list[Message]] = {}

    async def get_messages(
        self, session_id: str | None, *, state: dict[str, Any] | None = None, **kwargs: Any
    ) -> list[Message]:
        key = session_id or "default"
        return list(self._storage.get(key, []))

    async def save_messages(
        self,
        session_id: str | None,
        messages: Sequence[Message],
        *,
        state: dict[str, Any] | None = None,
        **kwargs: Any,
    ) -> None:
        key = session_id or "default"

        if key not in self._storage:
            self._storage[key] = []
        self._storage[key].extend(messages)

In [4]:
credential = AzureCliCredential()
client = OpenAIChatClient(
    model=model,
    azure_endpoint=azure_endpoint,
    credential=credential,
)

In [5]:
history_provider = CustomHistoryProvider()

agent = client.as_agent(
    name="MemoryBot",
    instructions="""You are a helpful assistant that remembers our conversation.
You have access to both conversation history and semantic memories from past interactions.
The memories from previous interactions are automatically provided to you.""",
    context_providers=[history_provider],
)

session = agent.create_session()

In [7]:
query = "Hello! My name is Kritika. I love pizza, prefer dark roast coffee, and I'm allergic to shellfish."

print(f"User: {query}")
print(f"Agent: {await agent.run(query, session=session)}\n")

User: Hello! My name is Kritika. I love pizza, prefer dark roast coffee, and I'm allergic to shellfish.
Agent: Hi again, Kritika! Got it—you love pizza, prefer dark roast coffee, and are allergic to shellfish. It seems like that’s something you really wanted me to remember, and I won’t forget it! How can I help you today? 😊



In [8]:
serialized_session = session.to_dict()

print(f"Serialized session: {serialized_session}\n")

resumed_session = AgentSession.from_dict(serialized_session)

Serialized session: {'type': 'session', 'session_id': '359e95ae-7bff-4e36-a230-5cd33d0cd34e', 'service_session_id': 'resp_02d370be59dff7fe0069e08ae8b95c819781547beb377ae79e', 'state': {'custom-history': {}}}



In [9]:
query = "What do you remember about my preferences? Can you recommend a snack for me?"

print(f"User: {query}")

agent = client.as_agent(
    name="MemoryBot",
    instructions="""You are a helpful assistant that remembers our conversation.
You have access to both conversation history and semantic memories from past interactions.
The memories from previous interactions are automatically provided to you.""",
    context_providers=[history_provider],
)

print(f"Agent: {await agent.run(query, session=resumed_session)}\n")

User: What do you remember about my preferences? Can you recommend a snack for me?
Agent: Of course, Kritika! Here's what I remember about your preferences:  
- You love pizza 🍕  
- You prefer dark roast coffee ☕  
- You're allergic to shellfish 🦐 (so we'll steer clear of anything that might include it).  

For snacks, I recommend:  
- **Garlic breadsticks or cheesy bread** (they complement your love for pizza perfectly).  
- **Dark chocolate-covered espresso beans** (to pair with your love for dark roast coffee).  
- **Veggie or chicken quesadillas** (flavorful and safe with no shellfish).  

Would you like more ideas? 😊

